In [1]:
## document structure

from langchain_core.documents import Document

In [2]:
doc = Document( 

    page_content="This is the content of the document.",
    metadata={ "source": "example.pdf", "author": "John Doe", "pages": 1,"date_created": "2023-10-01"}
)

In [3]:
doc

Document(metadata={'source': 'example.pdf', 'author': 'John Doe', 'pages': 1, 'date_created': '2023-10-01'}, page_content='This is the content of the document.')

In [4]:
## Create a simple text file 

import os

os.makedirs("../data/text_files", exist_ok=True)

In [5]:
sample_text ={

    "../data/text_files/python_intro.txt": """Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and vibrant community make it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. Its syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages."""


}

for file_path, content in sample_text.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

        print("semple text files created successfully!")

semple text files created successfully!


In [6]:
## Text Loader 

from langchain_community.document_loaders import TextLoader

from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()
print(documents)


C:\Users\tusharsharma.securit\AppData\Local\Temp\ipykernel_37440\2866035601.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\AgenticAIworkspace\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and vibrant community make it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. Its syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages.")]


In [7]:
## directory loader

from langchain_community.document_loaders import DirectoryLoader

## load all the text files in the directory

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False,
)

documents = dir_loader.load()

documents

[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and vibrant community make it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. Its syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages.")]

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
import os

## load all the text files in the directory

dir_loader = DirectoryLoader(
    "../data/New folder",
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,

    show_progress=False,
)

pdf_dir = "../data/pdf"
os.makedirs(pdf_dir, exist_ok=True)

try:
    documents = dir_loader.load()
    print(f"Loaded {len(documents)} documents from {pdf_dir}")
except Exception as e:
    print(f"Error loading PDFs from {pdf_dir}: {e}")
    documents = []

documents

Error loading PDFs from ../data/pdf: Directory not found: '../data/New folder'


[]

In [9]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
from pathlib import Path

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))#printing the length number of PDF files found

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"✗ Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#process all PDFs in the specified directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"✗ Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_directory = process_all_pdfs("../data/pdf")

Found 1 PDF files to process

Processing: ssrn-5980914.pdf
✓ Loaded 212 pages

Total documents loaded: 212


In [11]:
# Text splitting into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunk = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = globals().get("all_pdf_directory")
if docs is None:
    docs = globals().get("documents")
if docs is None:
 raise NameError("No documents are available. Run the PDF loading cell first.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

chunk = text_splitter.split_documents(docs)
chunk 

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-12-29T09:59:28+05:30', 'title': 'The Agentic AI Handbook', 'author': 'Kushal Bharadwaj', 'subject': 'Concepts, Design Patterns, and Future Directions', 'moddate': '2025-12-29T09:59:28+05:30', 'source': '..\\data\\pdf\\ssrn-5980914.pdf', 'total_pages': 212, 'page': 0, 'page_label': '1', 'source_file': 'ssrn-5980914.pdf', 'file_type': 'pdf'}, page_content='THE AGENTIC AI \nHANDBOOK \nConcepts, Design Patterns, and Future \nDirections \n \n \n \n \nKAUSHIK BAR'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-12-29T09:59:28+05:30', 'title': 'The Agentic AI Handbook', 'author': 'Kushal Bharadwaj', 'subject': 'Concepts, Design Patterns, and Future Directions', 'moddate': '2025-12-29T09:59:28+05:30', 'source': '..\\data\\pdf\\ssrn-5980914.pdf', 'total_pages': 212, 

### EMBEDDING And vectorestoredb


In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict,Tuple,Any
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class embedingManager:
    """Handles embedding generation using SentenceTransformers."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.

        Args:
            model_name (str): HuggingFace / sentence-transformers model name.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dim: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.

        Args:
            texts (List[str]): List of text strings to embed.

        Returns:
            np.ndarray: Array of embeddings.
        """
        if self.model is None:
            raise ValueError("Model not loaded. Call _load_model() first.")
        if isinstance(texts, str):
            texts = [texts]
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        return embeddings

    def get_embedding_dimension(self) -> int:
        """Return the embedding vector dimension."""
        if self.model is None:
            raise ValueError("Model not loaded. Call _load_model() first.")
        return self.model.get_sentence_embedding_dimension()

## initialize the embedding manager
embedding_manager = embedingManager()
embedding_manager

Loading model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6236.39it/s]


Model loaded successfully. Embedding dim: 384


C:\Users\tusharsharma.securit\AppData\Local\Temp\ipykernel_37440\3742389427.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dim: {self.model.get_sentence_embedding_dimension()}")


### Vectorestore


In [15]:
class VectoreStore:
    """Manager for vector store operations using ChromaDB."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/pdf_documents"):
        """Initialize the vector store manager."""
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG applications."},
            )
            print(f"vector store initialized. collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Document], embedding_manager: embedingManager):
        """Add documents to the vector store after generating embeddings."""
        if not documents:
            print("No documents to add.")
            return

        texts = [doc.page_content for doc in documents]
        embeddings = embedding_manager.generate_embeddings(texts)

        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")

        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embedding_list,
            )
            print(f"✓ Added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------
# STEP 1: Get loaded documents
# ---------------------------------------
docs = (
    globals().get("all_pdf_directory")
    or globals().get("documents")
    or globals().get("docs")
)

if docs is None:
    raise NameError(
        "No documents available. Run the PDF loading cell first."
    )

# ---------------------------------------
# STEP 2: Create Text Splitter
# ---------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

# ---------------------------------------
# STEP 3: Split Documents
# ---------------------------------------
chunk = text_splitter.split_documents(docs)

print(f"Total Chunks Created: {len(chunk)}")

# ---------------------------------------
# STEP 4: Extract Text
# ---------------------------------------
texts = [doc.page_content for doc in chunk]

# ---------------------------------------
# STEP 5: Generate Embeddings
# ---------------------------------------
embedding = embedding_manager.generate_embeddings(texts)

print(f"Generated {len(embedding)} embeddings")

# ---------------------------------------
# STEP 6: Create/Open Vector Store
# ---------------------------------------
vectorestore = VectoreStore(
    collection_name="pdf_documents",
    persist_directory="../data/vector_store"
)

# ---------------------------------------
# STEP 7: Store Documents
# ---------------------------------------
vectorestore.add_documents(chunk, embedding_manager)

print("Documents stored successfully!")

Total Chunks Created: 358
Generating embeddings for 358 texts...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches: 100%|██████████| 12/12 [00:12<00:00,  1.05s/it]


Generated 358 embeddings
vector store initialized. collection: pdf_documents
existing documents in collection: 0
Generating embeddings for 358 texts...


Batches: 100%|██████████| 12/12 [00:12<00:00,  1.04s/it]


✓ Added 358 documents to the vector store.
Total documents in collection: 358
Documents stored successfully!


### Retriver pipline for vectorestore

In [26]:
from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score
                    # (ChromaDB uses cosine distance)
                    retrieved_docs.append(
    {
        "id": doc_id,
        "content": document,
        "metadata": metadata,
        "distance": distance,
        "rank": i + 1,
    }
)
                print(
                    f"Retrieved {len(retrieved_docs)} documents (after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
   

In [27]:
# Instantiate or reuse existing instances before creating the retriever
vectorestore = globals().get("vectorestore") or VectoreStore()
embedding_manager = globals().get("embedding_manager") or embedingManager()
rag_retriever = RAGRetriever(vectorestore, embedding_manager)
print("RAG retriever initialized with VectoreStore and embedingManager instances")

RAG retriever initialized with VectoreStore and embedingManager instances


In [28]:
rag_retriever.retrieve("Core Concepts and Definitions")

Retrieving documents for query: 'Core Concepts and Definitions'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.05it/s]

Retrieved 5 documents (after filtering)


[{'id': 'doc_b19e6808_18',
  'content': 'Agency in humans vs. machines...................................... 28 \nTypes of AI agents ............................................................ 31 \nKey takeaways .................................................................... 33 \nChapter 3: Core Concepts and Definitions ....................... 34 \nGoals ................................................................................... 34 \nPlanning .............................................................................. 36 \nSelf-Reflection .................................................................... 37 \nSituatedness ........................................................................ 38 \nEmbodiment ...................................................................... 38 \nKey takeaways .................................................................... 39 \nPART II: Technical Building Blocks ........................... 41 \nChapter 4: Planning and Goal-Di

In [25]:
results = vectorestore.collection.query(
    query_texts=["Core Concepts and Definitions"],
    n_results=5
)

print(results)

C:\Users\tusharsharma.securit\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:14<00:00, 1.11MiB/s]


{'ids': [['doc_b19e6808_18', 'doc_f6a24d9b_31', 'doc_7e9db573_203', 'doc_a1a36113_121', 'doc_ee9b0d09_89']], 'embeddings': None, 'documents': [['Agency in humans vs. machines...................................... 28 \nTypes of AI agents ............................................................ 31 \nKey takeaways .................................................................... 33 \nChapter 3: Core Concepts and Definitions ....................... 34 \nGoals ................................................................................... 34 \nPlanning .............................................................................. 36 \nSelf-Reflection .................................................................... 37 \nSituatedness ........................................................................ 38 \nEmbodiment ...................................................................... 38 \nKey takeaways .................................................................... 

## Intigration vectoredb with  LLM 

In [ ]:
### simple RAG Pipeline with groq

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initalize the groq LLM 
groq_api_key="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"
 
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY
)
 
## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k=3):

    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = invoke(prompt)

    return response.content

In [39]:
answer=rag_simple("What is Core Concepts and Definitions?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Core Concepts and Definitions?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 91.96it/s]

Retrieved 3 documents (after filtering)


Chapter 3: Core Concepts and Definitions, which covers foundational concepts such as goals, planning, self-reflection, situatedness, and embodiment.
